# Пошаговое тестирование каждой функции модуля data

## Настройка окружения


In [1]:
import sys
from pathlib import Path

# Добавляем корень проекта в PYTHONPATH
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(f"Корень проекта: {PROJECT_ROOT}")

Корень проекта: D:\Projects\PythonProject\music-genre-classification-fma


In [2]:
import pandas as pd
import numpy as np

# Настройки для отображения
pd.set_option('display.max_columns', 10)
pd.set_option('display.max_rows', 10)

## Тестирование конфигурации и путей

In [3]:
from src.data.config import paths, audio_params, PROJECT_ROOT as SRC_ROOT

print("=== Тест 1: Конфигурация ===")
print(f"PROJECT_ROOT: {SRC_ROOT}")
print(f"\npaths.metadata_dir: {paths.metadata_dir}")
print(f"paths.active_zip: {paths.active_zip}")
print(f"paths.active_subset: {paths.active_subset}")

=== Тест 1: Конфигурация ===
PROJECT_ROOT: D:\Projects\PythonProject\music-genre-classification-fma

paths.metadata_dir: E:\music-genre-classifier\fma_metadata
paths.active_zip: E:\music-genre-classifier\fma_medium.zip
paths.active_subset: medium


In [4]:
print("=== Тест 2: Существование файлов ===")
tracks_path = paths.get_tracks_csv()
features_path = paths.get_features_csv()
genres_path = paths.get_genres_csv()

print(f"tracks.csv: {tracks_path}")
print(f"  Существует: {tracks_path.exists()}")
print(f"  Размер: {tracks_path.stat().st_size / 1024 / 1024:.2f} MB" if tracks_path.exists() else "  Файл не найден")

print(f"\nfeatures.csv: {features_path}")
print(f"  Существует: {features_path.exists()}")
print(f"  Размер: {features_path.stat().st_size / 1024 / 1024:.2f} MB" if features_path.exists() else "  Файл не найден")

print(f"\ngenres.csv: {genres_path}")
print(f"  Существует: {genres_path.exists()}")

=== Тест 2: Существование файлов ===
tracks.csv: E:\music-genre-classifier\fma_metadata\tracks.csv
  Существует: True
  Размер: 248.35 MB

features.csv: E:\music-genre-classifier\fma_metadata\features.csv
  Существует: True
  Размер: 907.06 MB

genres.csv: E:\music-genre-classifier\fma_metadata\genres.csv
  Существует: True


In [5]:
print("=== Тест 3: Аудио параметры ===")
print(f"sample_rate: {audio_params.sample_rate}")
print(f"duration: {audio_params.duration} sec")
print(f"n_mels: {audio_params.n_mels}")
print(f"n_mfcc: {audio_params.n_mfcc}")
print(f"hop_length: {audio_params.hop_length}")

=== Тест 3: Аудио параметры ===
sample_rate: 22050
duration: 30 sec
n_mels: 128
n_mfcc: 20
hop_length: 512


## Тестирование загрузчика FMALoader

In [6]:
from src.data.loader import FMALoader

print("=== Тест 4: Инициализация FMALoader ===")
loader = FMALoader()
print(f"loader.metadata_dir: {loader.metadata_dir}")

=== Тест 4: Инициализация FMALoader ===
loader.metadata_dir: E:\music-genre-classifier\fma_metadata


In [7]:
print("=== Тест 5: Загрузка tracks.csv ===")
try:
    tracks = loader.tracks
    print(f"✅ tracks загружен: {tracks.shape}")
    print(f"Индекс: {tracks.index.name}")
    print(f"Колонки (первые 5): {list(tracks.columns[:5])}")
    print(f"Тип мультииндекса: {tracks.columns.nlevels} уровня")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 5: Загрузка tracks.csv ===


KeyboardInterrupt: 

In [ ]:
print("=== Тест 5.1: Структура колонок ===")
for level in tracks.columns.levels:
    print(f"  Уровень '{level}':")
    print(f"    Значения: {list(level)}")

In [ ]:
print("=== Тест 5.2: Доступ к колонкам ===")
try:
    genre_col = tracks['track', 'genre_top']
    print(f"✅ Доступ к genre_top: первые 5 значений:\n{genre_col.head(5)}")
except KeyError as e:
    print(f"❌ Ошибка доступа: {e}")
    print("Попробуем альтернативный доступ...")
    try:
        genre_col = tracks['track']['genre_top']
        print(f"✅ Альтернативный доступ работает:\n{genre_col.head(5)}")
    except Exception as e2:
        print(f"❌ Альтернативный тоже не работает: {e2}")

In [ ]:
print("=== Тест 6: Загрузка features.csv ===")
try:
    features = loader.features
    print(f"✅ features загружены: {features.shape}")
    print(f"Колонки (первые 5): {list(features.columns[:5])}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 7: Загрузка genres.csv ===")
try:
    genres = loader.genres
    print(f"✅ genres загружены: {genres.shape}")
    print(f"genres.head():\n{genres.head()}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 8: Фильтрация по подмножеству ===")
try:
    medium_tracks = loader.get_tracks_by_subset('medium')
    print(f"✅ Треков в Medium: {len(medium_tracks)}")

    # Проверяем все ли треки из Medium
    subsets = medium_tracks['set', 'subset'].unique()
    print(f"Уникальные подмножества: {subsets}")
    assert len(subsets) == 1 and subsets[0] == 'medium', "Не все треки из Medium!"
    print("✅ Проверка пройдена: все треки из Medium")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 9: Доступные подмножества ===")
try:
    all_subsets = loader.tracks['set', 'subset'].unique()
    print(f"Доступные подмножества в данных: {all_subsets}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 10: Получение разбиения train/val/test ===")
try:
    splits = loader.get_available_splits(medium_tracks)
    print(f"✅ Разбиение получено:")
    print(f"  training: {len(splits['training'])} треков")
    print(f"  validation: {len(splits['validation'])} треков")
    print(f"  test: {len(splits['test'])} треков")

    total = sum(len(v) for v in splits.values())
    print(f"  Всего: {total} ({total/len(medium_tracks)*100:.1f}% от всех)")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 11: Маппинг жанров ===")
try:
    genre_mapping = loader.get_genre_mapping()
    print(f"✅ Жанров в маппинге: {len(genre_mapping)}")
    print(f"Примеры: {dict(list(genre_mapping.items())[:5])}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

## Тестирование препроцессора

In [ ]:
from src.data.preprocessor import DataPreprocessor

print("=== Тест 12: Инициализация препроцессора ===")
preprocessor = DataPreprocessor(min_samples_per_genre=100)
print(f"✅ Препроцессор создан")
print(f"  min_samples_per_genre: {preprocessor.min_samples_per_genre}")

In [ ]:
print("=== Тест 13: Фильтрация редких жанров ===")

genre_col = ('track', 'genre_top')
genre_counts = medium_tracks[genre_col].value_counts()
print(f"Жанров до фильтрации: {len(genre_counts)}")
print(f"Треков до фильтрации: {len(medium_tracks)}")

In [ ]:
print("=== Тест 13: Фильтрация редких жанров ===")

genre_col = ('track', 'genre_top')
genre_counts = medium_tracks[genre_col].value_counts()
print(f"Жанров до фильтрации: {len(genre_counts)}")
print(f"Треков до фильтрации: {len(medium_tracks)}")

filtered_tracks = preprocessor.filter_rare_genres(medium_tracks, genre_col)
print(f"\nЖанров после фильтрации: {len(filtered_tracks[genre_col].value_counts())}")
print(f"Треков после фильтрации: {len(filtered_tracks)}")

In [ ]:
print("=== Тест 14: Кодирование меток ===")

test_tracks = filtered_tracks.copy()
genre_col = ('track', 'genre_top')

train_mask = test_tracks['set', 'split'] == 'training'
val_mask = test_tracks['set', 'split'] == 'validation'
test_mask = test_tracks['set', 'split'] == 'test'

y_train = test_tracks[train_mask][genre_col]
y_val = test_tracks[val_mask][genre_col]
y_test = test_tracks[test_mask][genre_col]

print(f"y_train size: {len(y_train)}, y_val: {len(y_val)}, y_test: {len(y_test)}")

try:
    y_train_enc, y_val_enc, y_test_enc = preprocessor.encode_labels(y_train, y_val, y_test)
    print(f"✅ Кодирование выполнено")
    print(f"  Классов: {len(preprocessor.label_encoder.classes_)}")
    print(f"  Пример соответствия: {dict(zip(preprocessor.label_encoder.classes_, range(len(preprocessor.label_encoder.classes_))))}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 15: Нормализация признаков ===")

common_ids = test_tracks.index.intersection(features.index)
X = features.loc[common_ids]

X_train_raw = X.loc[test_tracks[train_mask].index]
X_val_raw = X.loc[test_tracks[val_mask].index]
X_test_raw = X.loc[test_tracks[test_mask].index]

print(f"X_train shape: {X_train_raw.shape}")
print(f"X_val shape: {X_val_raw.shape}")
print(f"X_test shape: {X_test_raw.shape}")

try:
    X_train_scaled, X_val_scaled, X_test_scaled = preprocessor.normalize_features(
        X_train_raw, X_val_raw, X_test_raw
    )
    print(f"✅ Нормализация выполнена")
    print(f"  X_train: mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
    print(f"  X_val:   mean={X_val_scaled.mean():.4f}, std={X_val_scaled.std():.4f}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 16: Веса классов ===")
try:
    weights = preprocessor.get_class_weights(y_train_enc)
    print(f"✅ Веса классов вычислены")
    print(f"  Веса: {weights}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 17: Сохранение препроцессора ===")
save_path = paths.processed_data_dir / 'test_preprocessor.pkl'
try:
    preprocessor.save(save_path)
    print(f"✅ Препроцессор сохранён: {save_path}")
    print(f"  Файл существует: {save_path.exists()}")
    print(f"  Размер: {save_path.stat().st_size} bytes")
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
print("=== Тест 18: Загрузка препроцессора ===")
new_preprocessor = DataPreprocessor()
try:
    new_preprocessor.load(save_path)
    print(f"✅ Препроцессор загружен")
    print(f"  min_samples_per_genre: {new_preprocessor.min_samples_per_genre}")
    print(f"  classes: {list(new_preprocessor.label_encoder.classes_)[:5]}...")
except Exception as e:
    print(f"❌ Ошибка: {e}")

## Тестирование работы с ZIP

In [ ]:
print("=== Тест 19: Проверка ZIP архива ===")
if paths.active_zip.exists():
    import zipfile
    try:
        with zipfile.ZipFile(paths.active_zip, 'r') as zf:
            files = zf.namelist()
            print(f"✅ ZIP открывается")
            print(f"  Файлов в архиве: {len(files)}")
            print(f"  MP3 файлов: {len([f for f in files if f.endswith('.mp3')])}")
            print(f"  Примеры: {files[:3]}")
    except Exception as e:
        print(f"❌ Ошибка: {e}")
else:
    print(f"⚠️ ZIP не найден: {paths.active_zip}")

## Итоговая проверка

In [ ]:
print("\n" + "=" * 60)
print("Итоговая сводка тестирования")
print("=" * 60)

checks = [
    ("Конфигурация загружена", True),
    ("tracks.csv существует", tracks_path.exists()),
    ("features.csv существует", features_path.exists()),
    ("FMALoader работает", 'loader' in dir()),
    ("Фильтрация Medium работает", len(medium_tracks) > 0),
    ("Разбиение получено", len(splits) == 3),
    ("Препроцессор работает", preprocessor is not None),
    ("Кодирование работает", 'y_train_enc' in dir()),
    ("Нормализация работает", 'X_train_scaled' in dir()),
]

all_passed = True
for name, status in checks:
    status_str = "✅" if status else "❌"
    print(f"{status_str} {name}")
    if not status:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("✅ ВСЕ ТЕСТЫ ПРОЙДЕНЫ! Можно запускать полный пайплайн.")
else:
    print("⚠️ НЕКОТОРЫЕ ТЕСТЫ НЕ ПРОЙДЕНЫ. Проверьте ошибки выше.")
print("=" * 60)

In [ ]:
import json

test_results = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'checks': {name: status for name, status in checks},
    'data_shapes': {
        'tracks': loader.tracks.shape if 'loader' in dir() and hasattr(loader, 'tracks') else None,
        'features': loader.features.shape if 'loader' in dir() and hasattr(loader, 'features') else None,
        'medium_tracks': len(medium_tracks) if 'medium_tracks' in dir() else None,
        'train_size': len(y_train) if 'y_train' in dir() else None,
    }
}

with open(paths.results_dir / 'test_results.json', 'w') as f:
    json.dump(test_results, f, indent=2, default=str)

print(f"✅ Результаты тестов сохранены в {paths.results_dir / 'test_results.json'}")

## ЗАПУСК ПОЛНОГО ПАЙПЛАЙНА

### Настройка окружения

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(f"Корень проекта: {PROJECT_ROOT}")

In [ ]:
from src.data.pipeline import DataPipeline
from src.data.load_processed import load_data, LoadProcessedData
from src.data.config import paths, audio_params

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Все модули импортированы")

### Проверка перед запуском

In [ ]:
print("=== Финальная проверка перед запуском ===")
print(f"Метаданные: {paths.metadata_dir}")
print(f"  tracks.csv: {paths.get_tracks_csv().exists()}")
print(f"  features.csv: {paths.get_features_csv().exists()}")
print(f"  genres.csv: {paths.get_genres_csv().exists()}")
print(f"\nАктивный датасет: {paths.active_subset}")
print(f"Активный ZIP: {paths.active_zip}")
print(f"  ZIP существует: {paths.active_zip.exists()}")

### Запуск пайплайна

In [ ]:
print("\n" + "="*60)
print("ЗАПУСК ПАЙПЛАЙНА")
print("="*60)

pipeline = DataPipeline(
    subset="medium",
    min_samples_per_genre=10,
    use_features=True,
    cache_dir=paths.processed_data_dir
)

print("\nПараметры пайплайна:")
print(f"  subset: {pipeline.subset}")
print(f"  min_samples_per_genre: {pipeline.min_samples_per_genre}")
print(f"  use_features: {pipeline.use_features}")
print(f"  cache_dir: {pipeline.cache_dir}")

In [ ]:
print("\n⏳ Запуск обработки данных (может занять 2-5 минут)...")
data = pipeline.run(force_reload=True)

In [ ]:
pipeline.print_summary()

### Анализ полученных данных

In [ ]:
print("\n=== Детальный анализ данных ===")

print(f"\nРазмеры выборок:")
print(f"  X_train: {data['X_train'].shape}")
print(f"  X_val:   {data['X_val'].shape}")
print(f"  X_test:  {data['X_test'].shape}")

print(f"\nТипы данных:")
print(f"  X_train dtype: {data['X_train'].dtype}")
print(f"  y_train dtype: {data['y_train'].dtype}")

print(f"\nСтатистика нормализации (проверка):")
print(f"  X_train mean: {data['X_train'].mean():.6f}, std: {data['X_train'].std():.6f}")
print(f"  X_val mean:   {data['X_val'].mean():.6f}, std: {data['X_val'].std():.6f}")
print(f"  X_test mean:  {data['X_test'].mean():.6f}, std: {data['X_test'].std():.6f}")

In [ ]:
print("\n=== Информация о метках ===")
print(f"Количество классов: {len(data['genre_names'])}")
print(f"Жанры: {data['genre_names']}")

unique, counts = np.unique(data['y_train'], return_counts=True)
print(f"\nРаспределение классов (train):")
for code, genre in zip(unique, data['genre_names']):
    count = counts[code]
    pct = count / len(data['y_train']) * 100
    bar = "█" * int(pct / 2)
    print(f"  {genre:20} {count:6} ({pct:5.1f}%) {bar}")

In [ ]:
print("\n=== Проверка на пропуски ===")
print(f"NaN в X_train: {np.isnan(data['X_train']).sum()}")
print(f"NaN в X_val:   {np.isnan(data['X_val']).sum()}")
print(f"NaN в X_test:  {np.isnan(data['X_test']).sum()}")
print(f"Inf в X_train: {np.isinf(data['X_train']).sum()}")

### Проверка кэширования

In [ ]:
print("\n=== Проверка кэширования ===")

cache_files = list(paths.processed_data_dir.glob("*"))
print(f"Файлы в кэше ({paths.processed_data_dir}):")
for f in cache_files:
    size = f.stat().st_size / 1024 / 1024
    print(f"  {f.name} ({size:.2f} MB)")

In [ ]:
print("\n⏳ Тестируем быструю загрузку из кэша...")
loader = LoadProcessedData(subset="medium", min_samples_per_genre=10)
if loader.exists():
    data_reloaded = loader.load()
    print(f"✅ Данные загружены из кэша")
    print(f"  X_train shape: {data_reloaded['X_train'].shape}")

    if np.allclose(data['X_train'], data_reloaded['X_train']):
        print("  ✅ Данные совпадают с оригиналом")
    else:
        print("  ⚠️ Данные НЕ совпадают!")
else:
    print("❌ Кэш не найден!")

# Поэтапное тестирование функционала XGBoost

## Импорт и настройка окружения

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.config import paths, audio_params

print("=" * 60)
print("ТЕСТИРОВАНИЕ КОНФИГУРАЦИИ ПУТЕЙ")
print("=" * 60)

print(f"\n📁 PROJECT_ROOT: {paths._get_dir('')}")
print(f"\n📂 Внешние данные:")
print(f"   metadata_dir: {paths.metadata_dir}")
print(f"   active_subset: {paths.active_subset}")

print(f"\n📂 Внутренние директории:")
print(f"   processed_data_dir: {paths.processed_data_dir}")
print(f"   results_dir: {paths.results_dir}")
print(f"   models_dir: {paths.models_dir}")

print(f"\n🎵 XGBoost Mono директории:")
print(f"   models: {paths.xgboost_mono_models_dir}")
print(f"   plots: {paths.xgboost_mono_plots_dir}")
print(f"   metrics: {paths.xgboost_mono_metrics_dir}")
print(f"   grid_search: {paths.xgboost_mono_grid_search_dir}")

print(f"\n🎛️ Аудио параметры:")
audio_params.print_info()

paths.ensure_dirs()
print("\n✅ Все директории созданы")

ТЕСТИРОВАНИЕ КОНФИГУРАЦИИ ПУТЕЙ

📁 PROJECT_ROOT: D:\Projects\PythonProject\music-genre-classification-fma

📂 Внешние данные:
   metadata_dir: E:\music-genre-classifier\fma_metadata
   active_subset: medium

📂 Внутренние директории:
   processed_data_dir: D:\Projects\PythonProject\music-genre-classification-fma\data\processed
   results_dir: D:\Projects\PythonProject\music-genre-classification-fma\results
   models_dir: D:\Projects\PythonProject\music-genre-classification-fma\models

🎵 XGBoost Mono директории:
   models: D:\Projects\PythonProject\music-genre-classification-fma\results\xgboost\mono\models
   plots: D:\Projects\PythonProject\music-genre-classification-fma\results\xgboost\mono\plots
   metrics: D:\Projects\PythonProject\music-genre-classification-fma\results\xgboost\mono\metrics
   grid_search: D:\Projects\PythonProject\music-genre-classification-fma\results\xgboost\mono\grid_search

🎛️ Аудио параметры:
[Аудио параметры]
  sample_rate:  22050
  duration:     30 sec
  n_mel

In [2]:
from src.data.load_processed import load_data, LoadProcessedData

print("=" * 60)
print("ТЕСТИРОВАНИЕ load_processed")
print("=" * 60)

data = load_data(subset="medium", min_samples_per_genre=10)

print(f"\n📊 Данные загружены:")
print(f"   X_train: {data['X_train'].shape}")
print(f"   X_val:   {data['X_val'].shape}")
print(f"   X_test:  {data['X_test'].shape}")
print(f"   y_train: {data['y_train'].shape}")
print(f"   genre_names: {data['genre_names']}")

print(f"\n📋 Метаданные:")
metadata = data['metadata']
print(f"   subset: {metadata['subset']}")
print(f"   num_classes: {metadata['num_classes']}")
print(f"   num_features: {metadata['num_features']}")
print(f"   train/val/test: {metadata['train_size']}/{metadata['val_size']}/{metadata['test_size']}")

print(f"\n📂 Проверка LoadProcessedData:")
loader_check = LoadProcessedData(subset="medium", min_samples_per_genre=10)
print(f"   Данные существуют: {loader_check.exists()}")
loader_check.print_info()

ТЕСТИРОВАНИЕ load_processed
Загрузка данных из D:\Projects\PythonProject\music-genre-classification-fma\data\processed...
✅ Загружены данные: 19922 train, 2505 val, 2573 test
   Жанров: 16

📊 Данные загружены:
   X_train: (19922, 518)
   X_val:   (2505, 518)
   X_test:  (2573, 518)
   y_train: (19922,)
   genre_names: ['Blues', 'Classical', 'Country', 'Easy Listening', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Jazz', 'Old-Time / Historic', 'Pop', 'Rock', 'Soul-RnB', 'Spoken']

📋 Метаданные:
   subset: medium
   num_classes: 16
   num_features: 518
   train/val/test: 19922/2505/2573

📂 Проверка LoadProcessedData:
   Данные существуют: True
ЗАГРУЖЕННЫЕ ДАННЫЕ
Датасет:        FMA MEDIUM
Треков:         25000
Признаков:      518
Жанров:         16

Разбиение:
  Train:        19922
  Val:          2505
  Test:         2573

Жанры: Blues, Classical, Country, Easy Listening, Electronic...


## Простое обучение

In [4]:
from src.models.xgboost_model import XGBoostGenreClassifier

print("=" * 60)
print("ТЕСТИРОВАНИЕ XGBoost МОДЕЛИ")
print("=" * 60)

data = load_data(subset="medium", min_samples_per_genre=10)

X_train, X_val, X_test = data['X_train'], data['X_val'], data['X_test']
y_train, y_val, y_test = data['y_train'], data['y_val'], data['y_test']
genre_names = data['genre_names']

print(f"\n📊 Данные для обучения:")
print(f"   Train: {X_train.shape}")
print(f"   Val:   {X_val.shape}")
print(f"   Test:  {X_test.shape}")

model = XGBoostGenreClassifier(
    params={
        'n_estimators': 50,
        'max_depth': 3,
        'learning_rate': 0.3,
    },
    use_class_weights=True
)

print("\n🔧 Обучение модели...")
model.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    genre_names=genre_names,
    verbose=True
)

print("\n💾 Сохранение модели...")
model.save(name="xgboost_first")

print("\n✅ Модель успешно обучена и сохранена!")

ТЕСТИРОВАНИЕ XGBoost МОДЕЛИ
Загрузка данных из D:\Projects\PythonProject\music-genre-classification-fma\data\processed...
✅ Загружены данные: 19922 train, 2505 val, 2573 test
   Жанров: 16

📊 Данные для обучения:
   Train: (19922, 518)
   Val:   (2505, 518)
   Test:  (2573, 518)

🔧 Обучение модели...
Обучение XGBoost классификатора
Треков train: 19922
Признаков: 518
Классов: 16
Веса классов: True
Параметры: {'n_estimators': 50, 'max_depth': 3, 'learning_rate': 0.3, 'random_state': 42, 'objective': 'multi:softmax', 'num_class': 16}
------------------------------------------------------------
[0]	validation_0-mlogloss:2.43759
[1]	validation_0-mlogloss:2.25906
[2]	validation_0-mlogloss:2.12913
[3]	validation_0-mlogloss:2.03721
[4]	validation_0-mlogloss:1.95333
[5]	validation_0-mlogloss:1.88783
[6]	validation_0-mlogloss:1.83670
[7]	validation_0-mlogloss:1.78799
[8]	validation_0-mlogloss:1.74811
[9]	validation_0-mlogloss:1.70860
[10]	validation_0-mlogloss:1.67490
[11]	validation_0-mlogloss:

In [5]:
print("=" * 60)
print("ТЕСТИРОВАНИЕ КОМПЛЕКСНОЙ ОЦЕНКИ")
print("=" * 60)

metrics = model.comprehensive_evaluate(X_test, y_test, genre_names)

print(f"\n📊 Результаты комплексной оценки:")
print(f"   Accuracy:      {metrics['accuracy']:.4f}")
print(f"   F1-macro:      {metrics['f1_macro']:.4f}")
print(f"   F1-weighted:   {metrics['f1_weighted']:.4f}")
print(f"   F1-micro:      {metrics['f1_micro']:.4f}")

print(f"\n   Top-1 Accuracy: {metrics['top_1_accuracy']:.4f}")
print(f"   Top-3 Accuracy: {metrics['top_3_accuracy']:.4f}")
print(f"   Top-5 Accuracy: {metrics['top_5_accuracy']:.4f}")

print(f"\n   Composite Score: {metrics['composite_score']:.4f}")

print(f"\n🎯 Анализ уверенности:")
conf = metrics['confidence']
print(f"   Уверенность (прав.): {conf['confidence_correct_mean']:.3f} ± {conf['confidence_correct_std']:.3f}")
print(f"   Уверенность (ошиб.): {conf['confidence_wrong_mean']:.3f} ± {conf['confidence_wrong_std']:.3f}")
print(f"   Разрыв:              {conf['confidence_gap']:.3f}")
print(f"   Низкая уверенность:  {conf['low_confidence_rate']:.2%}")

ТЕСТИРОВАНИЕ КОМПЛЕКСНОЙ ОЦЕНКИ

📊 Результаты комплексной оценки:
   Accuracy:      0.5705
   F1-macro:      0.3960
   F1-weighted:   0.5832
   F1-micro:      0.5705

   Top-1 Accuracy: 0.5705
   Top-3 Accuracy: 0.8053
   Top-5 Accuracy: 0.8982

   Composite Score: 0.5972

🎯 Анализ уверенности:
   Уверенность (прав.): 0.623 ± 0.220
   Уверенность (ошиб.): 0.413 ± 0.166
   Разрыв:              0.210
   Низкая уверенность:  49.71%


In [ ]:
from src.training.grid_search import XGBoostGridSearch, GRID_SMALL

print("=" * 60)
print("ТЕСТИРОВАНИЕ GRID SEARCH")
print("=" * 60)

data = load_data(subset="medium", min_samples_per_genre=10)

X_train, X_val, X_test = data['X_train'], data['X_val'], data['X_test']
y_train, y_val, y_test = data['y_train'], data['y_val'], data['y_test']
genre_names = data['genre_names']

print("\n🔧 Запуск Grid Search (маленькая сетка)...")

grid_search = XGBoostGridSearch(
    param_grid=GRID_SMALL,
    use_class_weights=True,
    verbose=True
)

results = grid_search.fit(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    genre_names
)

print("\n📊 Лучшие результаты:")
print(results.head(5).to_string())

grid_search.save_results()
print("\n✅ Grid Search завершён!")

ТЕСТИРОВАНИЕ GRID SEARCH
Загрузка данных из D:\Projects\PythonProject\music-genre-classification-fma\data\processed...
✅ Загружены данные: 19922 train, 2505 val, 2573 test
   Жанров: 16

🔧 Запуск Grid Search (маленькая сетка)...
XGBOOST GRID SEARCH (COMPREHENSIVE METRICS)
Всего комбинаций: 18
Метрики оценки: F1-macro, Top-3 Acc, F1-weighted, Composite Score
----------------------------------------------------------------------

[1/18] Тестирование: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}
  F1-macro: 0.3665 | Top-3: 0.7676 | Composite: 0.5632 | Time: 38.7s

[2/18] Тестирование: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  F1-macro: 0.3978 | Top-3: 0.7944 | Composite: 0.5939 | Time: 67.9s

[3/18] Тестирование: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 150}
  F1-macro: 0.4080 | Top-3: 0.8045 | Composite: 0.6027 | Time: 96.9s

[4/18] Тестирование: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 50}
  F1-macro: 0.4130 | Top-3: 0.8049 |